In [6]:
import torch
import os
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings("ignore")

MODEL_PATH = "./saved_model1/model1.pth" 
TOKENIZER_PATH = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
DATA_PATH = "dataset.csv"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Saved model not found at {MODEL_PATH}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

data = pd.read_csv(DATA_PATH).dropna(subset=['symptoms'])

checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
label_classes = checkpoint['label_encoder_classes']

import torch.serialization
torch.serialization.add_safe_globals(['_reconstruct'])

label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(label_classes)  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(TOKENIZER_PATH, num_labels=len(label_encoder.classes_))
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"Successfully loaded model with {len(label_encoder.classes_)} labels.")

def get_disease_embedding(disease_name, tokenizer, model):

    inputs = tokenizer(disease_name, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
      
        outputs = model(**inputs, output_hidden_states=True)
    
        last_hidden_state = outputs.hidden_states[-1]
        cls_embedding = last_hidden_state[:, 0, :]  
    
  
    return cls_embedding.cpu().numpy()

def calculate_semantic_similarity(pred_disease, true_disease, tokenizer, model):

    try:
        pred_embedding = get_disease_embedding(pred_disease, tokenizer, model)
        true_embedding = get_disease_embedding(true_disease, tokenizer, model)
        
        similarity = cosine_similarity(pred_embedding, true_embedding)[0][0]
        return max(0.0, min(1.0, similarity))  
    except Exception as e:
        print(f"Warning: Error calculating semantic similarity: {e}")
        return 0.0  

def calculate_probability_similarity(probs, true_label_idx):
  
    return probs[true_label_idx].item()

train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)


test_samples = test_data.sample(n=10, random_state=np.random.randint(0, 10000))  # Randomize on each run

correct_predictions = 0
total_samples = len(test_samples)
total_semantic_similarity = 0
total_probability_similarity = 0

for _, row in test_samples.iterrows():
    user_input = row['symptoms']
    true_label = row['diseases']
    true_label_idx = label_encoder.transform([true_label])[0]

    inputs = tokenizer(user_input, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1).flatten()
    predicted_index = torch.argmax(probs).item()
    predicted_disease = label_encoder.inverse_transform([predicted_index])[0]

    semantic_sim = calculate_semantic_similarity(predicted_disease, true_label, tokenizer, model)
    prob_sim = calculate_probability_similarity(probs, true_label_idx)
    

    total_semantic_similarity += semantic_sim
    total_probability_similarity += prob_sim



    if predicted_disease == true_label:
        correct_predictions += 1

accuracy = (correct_predictions / total_samples) * 100
avg_semantic_similarity = total_semantic_similarity / total_samples
avg_probability_similarity = total_probability_similarity / total_samples
similarity = (avg_semantic_similarity + avg_probability_similarity) / 2


print(f"Similarity Score: {similarity:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Successfully loaded model with 2401 labels.
Similarity Score: 0.7056
